In [ ]:
!wget http://www.inf.ufpr.br/vri/databases/PKLot.tar.gz

--2023-03-03 16:23:32--  http://www.inf.ufpr.br/vri/databases/PKLot.tar.gz
Resolving www.inf.ufpr.br (www.inf.ufpr.br)... 200.17.202.113, 2801:82:80ff:8001:216:ccff:feaa:79
Connecting to www.inf.ufpr.br (www.inf.ufpr.br)|200.17.202.113|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.inf.ufpr.br/vri/databases/PKLot.tar.gz [following]
--2023-03-03 16:23:34--  https://www.inf.ufpr.br/vri/databases/PKLot.tar.gz
Connecting to www.inf.ufpr.br (www.inf.ufpr.br)|200.17.202.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4898276304 (4.6G) [application/octet-stream]
Saving to: ‘PKLot.tar.gz’

PKLot.tar.gz        100%[===================>]   4.56G  14.4MB/s    in 5m 47s  

2023-03-03 16:29:22 (13.5 MB/s) - ‘PKLot.tar.gz’ saved [4898276304/4898276304]



In [ ]:
!tar xvzf  /content/PKLot.tar.gz

Streaming output truncated to the last 5000 lines.
PKLot/PKLotSegmented/PUC/Sunny/2012-09-20/Occupied/2012-09-20_18_09_45#088.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-09-28/Occupied/2012-09-28_09_06_05#086.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-10-31/Occupied/2012-10-31_14_33_21#092.jpg
PKLot/PKLotSegmented/UFPR04/Rainy/2013-01-21/Occupied/2013-01-21_08_35_04#011.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-10-16/Empty/2012-10-16_06_26_44#050.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-09-28/Occupied/2012-09-28_11_06_11#047.jpg
PKLot/PKLotSegmented/PUC/Sunny/2012-09-17/Occupied/2012-09-17_10_59_02#100.jpg
PKLot/PKLotSegmented/UFPR05/Sunny/2013-03-10/Empty/2013-03-10_17_35_13#027.jpg
PKLot/PKLotSegmented/PUC/Sunny/2012-10-27/Empty/2012-10-27_12_15_53#018.jpg
PKLot/PKLotSegmented/PUC/Rainy/2012-09-21/Occupied/2012-09-21_11_45_24#077.jpg
PKLot/PKLotSegmented/PUC/Sunny/2012-09-20/Occupied/2012-09-20_17_34_43#052.jpg
PKLot/PKLotSegmented/UFPR05/Cloudy/2013-03-17/Empty/2013-03-17_15_20_10#033.jpg

In [ ]:
!mkdir all_lots  all_lots/train_lots  all_lots/train_lots/occupied  all_lots/train_lots/vacant
!mkdir all_lots/test_lots  all_lots/test_lots/occupied all_lots/test_lots/vacant

In [ ]:
#opereat on Empty or occupied parking space (division)
def split_data(Empty_directory_p,occupied_directory_p,path_test_vacant,path_test_occupied,path_train_vacant,path_train_occupied):
  import shutil , os
  from random import shuffle

  for empty  in Empty_directory_p:
      E=os.listdir(empty)
      shuffle(E)
      E1=round(len(E)*.80)
      empty_train=E[:E1]
      empty_test=E[E1:]

      for empty_Train_1  in empty_train:
        shutil.move(empty+'/'+empty_Train_1,path_train_vacant)
        
      for empty_test_1 in empty_test:
        shutil.move(empty+'/'+empty_test_1,path_test_vacant)

  #===================================================================================
  for occupied in occupied_directory_p:
    O=os.listdir(occupied)
    shuffle(O)
    O1=round(len(O)*.80)
    occupied_train=O[:O1]
    occupied_test=O[O1:]    

    for  occupied_train_1 in occupied_train : 
        shutil.move(occupied+'/'+occupied_train_1,path_train_occupied)

    for occupied_test_1 in occupied_test:
        shutil.move(occupied+'/'+occupied_test_1,path_test_occupied)

  #===========================================================================================



In [ ]:
#prepare pathes for data handling 
import os 
views=os.listdir('/content/PKLot/PKLotSegmented')
weathers=os.listdir('/content/PKLot/PKLotSegmented/PUC')

#===========================================================
directorys=[]
for view in views:
  for weather in weathers:
    directorys.append(os.path.join(view,weather))
%cd /content/PKLot/PKLotSegmented


Empty_directory=[]
Occupied_directory=[]
#==================================================================
for dir in directorys:
  temp_list=os.listdir(dir)
  for temp in temp_list:
    two_class=os.listdir(dir+'/'+temp)
    if len(two_class)==2:
      Empty_directory.append(os.path.join(dir,temp+'/Empty'))
      Occupied_directory.append(os.path.join(dir,temp+'/Occupied'))
    elif two_class[0]=='Empty':
      Empty_directory.append(os.path.join(dir,temp+'/Empty'))
    elif two_class[0]=='Occupied':
      Occupied_directory.append(os.path.join(dir,temp+'/Occupied'))

#======================================================================
#REMOVE TWO DIRECTORY THAT CONTAIN SINGLITON PHOTO
Empty_directory.remove('UFPR05/Rainy/2013-02-26/Empty')
Occupied_directory.remove('UFPR04/Rainy/2012-12-15/Occupied')
#=======================================================================

split_data(Empty_directory,Occupied_directory,'/content/all_lots/test_lots/vacant','/content/all_lots/test_lots/occupied',
           '/content/all_lots/train_lots/vacant','/content/all_lots/train_lots/occupied')


/content/PKLot/PKLotSegmented


In [ ]:
!mkdir /content/train_data  /content/train_data/vacant /content/train_data/occupied

In [ ]:
def move_to_final_train_or_test_file(path_to_file_data_O,path_to_file_data_v,path_to_file_O,path_to_file_v):
  dirs_o = os.listdir(path_to_file_data_O)
  dirs_v= os.listdir(path_to_file_data_v)
  import shutil
  os.chdir(path_to_file_data_O)

  for dir in dirs_o:
      shutil.move(dir,path_to_file_O)
  
  os.chdir(path_to_file_data_v)
  for dir in dirs_v:
    shutil.move(dir,path_to_file_v)
  os.chdir('/content')


#print( len(os.listdir('/content/train_data/vacant')) + len(os.listdir('/content/train_data/occupied')) )

In [ ]:
move_to_final_train_or_test_file('/content/all_lots/train_lots/occupied','/content/all_lots/train_lots/vacant',
                         '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
print(len(os.listdir('/content/train_data/occupied'))+len(os.listdir('/content/train_data/vacant')))

556684


In [ ]:
%cd /content
#download CNR-EXT dataset and unzip it 
!wget http://claudiotest.isti.cnr.it/park-datasets/CNR-EXT/CNR-EXT-Patches-150x150.zip
!unzip /content/CNR-EXT-Patches-150x150.zip

Streaming output truncated to the last 5000 lines.
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_11.06_C08_211.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_08.06_C08_214.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_10.36_C08_325.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_16.36_C08_291.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_10.06_C08_283.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_08.36_C08_317.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_15.06_C08_321.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_11.36_C08_255.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_09.36_C08_292.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_11.36_C08_327.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_16.06_C08_324.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera

In [ ]:
!mkdir CNR-EXT
!mkdir CNR-EXT/OverCast/
!mkdir CNR-EXT/Rainy/
!mkdir CNR-EXT/Sunnay/

In [ ]:
import os
dirs=os.listdir('/content/CNR-EXT')
%cd '/content/CNR-EXT'

for dir in dirs:
  for i in range(1,10):
      inner_path=os.path.join( dir,"C-{}".format(i) )
      os.mkdir(inner_path)
      vacent_path=os.path.join(inner_path,'vacent')
      occupied_path=os.path.join(inner_path,'occupied')
      os.mkdir(vacent_path)
      os.mkdir(occupied_path)
 


/content/CNR-EXT


In [ ]:
def move_data(path_label,number_of_camera):
  import shutil
  
  ref_to_file=open(path_label,'r')

  for line in ref_to_file:
    l=line.split(" ")

    if not l[0].find('SUNNY'):
      if  int(l[1])==1:
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Sunnay/C-{}/occupied'.format(number_of_camera))
      else: 
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Sunnay/C-{}/vacent'.format(number_of_camera))
    elif not l[0].find('RAINY'):
      if int(l[1])==1:
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Rainy/C-{}/occupied'.format(number_of_camera))
      else: 
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Rainy/C-{}/vacent'.format(number_of_camera))
    else:
      if int(l[1])==1:
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/OverCast/C-{}/occupied'.format(number_of_camera))
      else: 
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/OverCast/C-{}/vacent'.format(number_of_camera))



In [ ]:
path_data='/content/PATCHES'

%cd '/content/PATCHES'
for i in range(1,10):
  path_label='/content/LABELS/camera{}.txt'.format(i)
  print(path_label)  
  move_data(path_label,i)


/content/PATCHES
/content/LABELS/camera1.txt
/content/LABELS/camera2.txt
/content/LABELS/camera3.txt
/content/LABELS/camera4.txt
/content/LABELS/camera5.txt
/content/LABELS/camera6.txt
/content/LABELS/camera7.txt
/content/LABELS/camera8.txt
/content/LABELS/camera9.txt


In [ ]:
%cd /content

/content


In [ ]:
!mkdir cnr-ext

In [ ]:
!mkdir cnr-ext/train cnr-ext/test
!mkdir cnr-ext/train/vacant cnr-ext/train/occupied 
!mkdir cnr-ext/test/vacant cnr-ext/test/occupied 

In [ ]:
dirs=os.listdir('/content/CNR-EXT')
%cd '/content/CNR-EXT'

Empty_directory_cnr=[]
occupied_directory_cnr=[]

for dir in dirs:
  for i in range(1,10):
      inner_path=os.path.join( dir,"C-{}".format(i) )
      vacent_path=os.path.join(inner_path,'vacent')
      occupied_path=os.path.join(inner_path,'occupied')
      Empty_directory_cnr.append('/content/CNR-EXT/'+vacent_path)
      occupied_directory_cnr.append('/content/CNR-EXT/'+occupied_path)   

/content/CNR-EXT


In [ ]:
split_data(Empty_directory_cnr,occupied_directory_cnr,'/content/cnr-ext/test/vacant','/content/cnr-ext/test/occupied',
           '/content/cnr-ext/train/vacant','/content/cnr-ext/train/occupied')

In [ ]:
move_to_final_train_or_test_file('/content/cnr-ext/train/occupied','/content/cnr-ext/train/vacant',
                         '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
print(len(os.listdir('/content/train_data/occupied'))+len(os.listdir('/content/train_data/vacant')))

672656


In [ ]:
#download CNR 
!wget http://claudiotest.isti.cnr.it/park-datasets/CNRPark/CNRPark-Patches-150x150.zip
!unzip /content/CNRPark-Patches-150x150.zip

Streaming output truncated to the last 5000 lines.
  inflating: B/busy/20150708_0855_1.jpg  
  inflating: B/busy/20150708_1620_25.jpg  
  inflating: B/busy/20150708_1240_21.jpg  
  inflating: B/busy/20150708_0950_48.jpg  
  inflating: B/busy/20150708_1705_4.jpg  
  inflating: B/busy/20150708_1400_21.jpg  
  inflating: B/busy/20150708_1300_14.jpg  
  inflating: B/busy/20150708_1625_11.jpg  
  inflating: B/busy/20150708_1505_50.jpg  
  inflating: B/busy/20150708_1720_15.jpg  
  inflating: B/busy/20150708_1640_45.jpg  
  inflating: B/busy/20150708_1105_42.jpg  
  inflating: B/busy/20150708_1150_35.jpg  
  inflating: B/busy/20150708_1230_53.jpg  
  inflating: B/busy/20150708_1255_38.jpg  
  inflating: B/busy/20150708_1140_3.jpg  
  inflating: B/busy/20150708_1140_31.jpg  
  inflating: B/busy/20150708_1040_22.jpg  
  inflating: B/busy/20150708_1140_6.jpg  
  inflating: B/busy/20150708_1150_41.jpg  
  inflating: B/busy/20150708_0945_47.jpg  
  inflating: B/busy/20150708_1105_50.jpg  
  infla

In [ ]:
!mkdir cnr

In [ ]:
!mkdir cnr/test cnr/train 

In [ ]:
!mkdir cnr/test/vacant cnr/test/occupied 
!mkdir cnr/train/vacant cnr/train/occupied 

In [ ]:
Empty_dir_cnr=['/content/A/free','/content/B/free']
Occupied_dir_cnr=['/content/A/busy','/content/B/busy']

split_data(Empty_dir_cnr,Occupied_dir_cnr,'/content/cnr/test/vacant',
           '/content/cnr/test/occupied','/content/cnr/train/vacant','/content/cnr/train/occupied')


In [ ]:
move_to_final_train_or_test_file('/content/cnr/train/occupied','/content/cnr/train/vacant',
                                 '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
print(len(os.listdir('/content/train_data/occupied'))+len(os.listdir('/content/train_data/vacant')))

682724


In [ ]:
!mkdir train 
!mkdir validation
!mkdir train/occupied train/vacant
!mkdir validation/occupied validation/vacant

In [ ]:
path_to_v=['/content/train_data/vacant',]
path_to_o=['/content/train_data/occupied',]
split_data(path_to_v,
          path_to_o,
          '/content/validation/vacant',
          '/content/validation/occupied',
          '/content/train/vacant',
          '/content/train/occupied')

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
def generate_data(path_to_data_dir):
    my_gen=ImageDataGenerator(1./255)
    
    data_gen=my_gen.flow_from_directory(
        path_to_data_dir,
        target_size=(150,150),
        batch_size=128,
        class_mode='categorical'
    )
    return data_gen

In [ ]:
def generate_data_1(path_to_data_dir):
  
  return tf.keras.utils.image_dataset_from_directory(
      path_to_data_dir,
  
      label_mode='categorical' ,
       
      color_mode='rgb',
      batch_size=128,
      image_size=(150, 150),
       shuffle=True,
    
  )

In [ ]:
from keras import layers ,models 
model = models.Sequential()

model.add(layers.Conv2D(32,(3,3),activation='relu',input_shape=(150,150,3)))
model.add(layers.MaxPool2D((2,2)))
model.add(layers.Conv2D(32,(3,3),activation='relu'))
model.add(layers.MaxPool2D((2,2)))
model.add(layers.Flatten())
model.add(layers.Dense(64,activation='relu'))
model.add(layers.Dense(2,activation='softmax'))

In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 72, 72, 32)        9248      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 36, 36, 32)       0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 41472)             0         
                                                                 
 dense (Dense)               (None, 64)                2

In [ ]:
model.compile(optimizer='Adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

In [ ]:
history=model.fit_generator(
      generate_data('train'),
      epochs=10,
      steps_per_epoch=546179/128,
      validation_data=generate_data('validation'),
      validation_steps=136545/128
      )

Found 546179 images belonging to 2 classes.
Found 136545 images belonging to 2 classes.


<ipython-input-37-351540e82f4a>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history=model.fit_generator(
/usr/local/lib/python3.8/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


Epoch 1/10
4267/4267 [==============================] - 1285s 298ms/step - loss: 0.4869 - accuracy: 0.9801 - val_loss: 0.0340 - val_accuracy: 0.9902
Epoch 2/10
4267/4267 [==============================] - 1137s 266ms/step - loss: 0.1316 - accuracy: 0.9603 - val_loss: 0.0896 - val_accuracy: 0.9727
Epoch 3/10
4267/4267 [==============================] - 1012s 237ms/step - loss: 0.0706 - accuracy: 0.9781 - val_loss: 0.0555 - val_accuracy: 0.9836
Epoch 4/10
4267/4267 [==============================] - 905s 212ms/step - loss: 0.0428 - accuracy: 0.9875 - val_loss: 0.0502 - val_accuracy: 0.9866
Epoch 5/10
4267/4267 [==============================] - 892s 209ms/step - loss: 0.0288 - accuracy: 0.9918 - val_loss: 0.0329 - val_accuracy: 0.9918
Epoch 6/10
4267/4267 [==============================] - 930s 218ms/step - loss: 0.0300 - accuracy: 0.9917 - val_loss: 0.0388 - val_accuracy: 0.9905
Epoch 7/10
4267/4267 [==============================] - 923s 216ms/step - loss: 0.0189 - accuracy: 0.9947 - v

In [ ]:
from google.colab import drive
drive.mount('/content/drive/my_model')

Mounted at /content/drive/my_model


In [ ]:
model.save('/content/drive/my_model car.h5')

In [ ]:
history_2=model.fit_generator(
      generate_data('train'),
      epochs=1,
      steps_per_epoch=546179/128,
      validation_data=generate_data('validation'),
      validation_steps=136545/128
      )

Found 546179 images belonging to 2 classes.
Found 136545 images belonging to 2 classes.


<ipython-input-50-edf1936c9501>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history_2=model.fit_generator(
/usr/local/lib/python3.8/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


4267/4267 [==============================] - 934s 219ms/step - loss: 0.0119 - accuracy: 0.9971 - val_loss: 0.0318 - val_accuracy: 0.9939


In [ ]:
model.save('/content/drive/my_model car_1.h5')

In [ ]:
history_3=model.fit_generator(
      generate_data('train'),
      epochs=1,
      steps_per_epoch=546179/128,
      validation_data=generate_data('validation'),
      validation_steps=136545/128
      )

Found 546179 images belonging to 2 classes.
Found 136545 images belonging to 2 classes.


<ipython-input-53-457e05de5c3b>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history_3=model.fit_generator(
/usr/local/lib/python3.8/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


4267/4267 [==============================] - 914s 214ms/step - loss: 0.0107 - accuracy: 0.9974 - val_loss: 0.0310 - val_accuracy: 0.9946


In [ ]:
model.save('/content/drive/my_model car_2.h5')

In [ ]:
history_4=model.fit_generator(
      generate_data('train'),
      epochs=1,
      steps_per_epoch=546179/128,
      validation_data=generate_data('validation'),
      validation_steps=136545/128
      )

Found 546179 images belonging to 2 classes.
Found 136545 images belonging to 2 classes.


<ipython-input-55-a6b102536bda>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history_4=model.fit_generator(
/usr/local/lib/python3.8/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


4267/4267 [==============================] - 947s 222ms/step - loss: 0.0111 - accuracy: 0.9974 - val_loss: 0.0376 - val_accuracy: 0.9929


In [ ]:
model.save('/content/drive/my_model car_3.h5')

In [ ]:
move_to_final_train_or_test_file('/content/train/occupied','/content/train/vacant',
                                 '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
move_to_final_train_or_test_file('/content/validation/occupied','/content/validation/vacant',
                                 '/content/train_data/occupied','/content/train_data/vacant')


In [ ]:
len(os.listdir('/content/train_data/occupied'))+len(os.listdir('/content/train_data/vacant'))

682724